In [28]:
import os
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance


In [29]:
VAULT_PATH = "/Users/amit/Obsidian"
COLLECTION_NAME = "obsidian_notes"

In [30]:

# 1. Init model
model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Init Qdrant
client = QdrantClient("localhost", port=6333)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7010.24it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [31]:
# 3. Create collection (run once)
client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

/var/folders/tj/swlqfzd11j9c_whnx01nv15w0000gn/T/ipykernel_43367/442226649.py:2: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [32]:
# 4. Chunking function
def chunk_text(text, chunk_size=200, overlap=50):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)

    return chunks


In [33]:

# 5. Read all .md files and prepare data
data = []

for root, _, files in os.walk(VAULT_PATH):
    for file in files:
        if file.endswith(".md"):
            file_path = os.path.join(root, file)

            with open(file_path, "r", encoding="utf-8") as f:
                content = f.read()

            chunks = chunk_text(content)

            for i, chunk in enumerate(chunks):
                data.append({
                    "text": chunk,
                    "payload": {
                        "source": file,
                        "path": file_path,
                        "chunk_id": i,
                        "type": "note"
                    }
                })

print(f"Total chunks: {len(data)}")

Total chunks: 1568


In [34]:
# 6. Generate embeddings (batched)
texts = [item["text"] for item in data]
embeddings = model.encode(texts, batch_size=32)

# 7. Insert into Qdrant
points = [
    PointStruct(
        id=i,
        vector=embeddings[i].tolist(),
        payload=data[i]["payload"] | {"text": data[i]["text"]}
    )
    for i in range(len(data))
]

In [35]:
print(data[0])

{'text': 'You are building a React Native clone of the "I Love Hue" color puzzle game. The app is called "HueShift". Build it in a strictly modular, scalable manner. Follow every instruction precisely. --- ## TECH STACK - React Native (Expo managed workflow) - React Navigation v6 (Stack + Bottom behavior where needed) - Zustand for global state (lives, currency, settings, level progress) - React Native Reanimated 3 for all animations - expo-linear-gradient for backgrounds and tile gradients - react-native-gesture-handler for drag interactions (gameplay phase) - TypeScript throughout — strict mode --- ## PROJECT STRUCTURE Organize the project exactly as follows: src/ components/ layout/ AppHeader.tsx ← Shared fixed header (lives + currency + settings icon) GameHeader.tsx ← Gameplay-only header (settings icon only, minimal) SettingsDropdown.tsx ← Animated dropdown, NOT a screen ui/ LivesCounter.tsx CurrencyBadge.tsx ModeCard.tsx ← Used on Home screen LevelCard.tsx ← Used on Level Select 

In [36]:
client.upsert(
    collection_name=COLLECTION_NAME,
    points=points
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [37]:
def search(query):
    query_vector = model.encode(query)

    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=5
    )

    for r in results.points:
        print(r.payload["source"])
        print(r.payload["text"])
        print(r.payload["path"])
        print(r.payload["chunk_id"])
        print("----")

In [39]:
search("what was the list with The Empire of the Sun as an item?")

Modern Black & White Films.md
- [ ] Francis Ha - [ ] Schindler's List - [ ] Roma - [ ] Cold War - [ ] Lighthouse
/Users/amit/Obsidian/Google Keep/Modern Black & White Films.md
0
----
War_Holocaust films.md
--- aliases: - War/Holocaust films --- - [ ] The Thin Red Line - [ ] The Pianist - [ ] Hacksaw Ridge - [ ] Schindler's List - [ ] Saving Private Ryan - [ ] 1917 - [ ] Fury - [ ] Life Is Beautiful - [ ] Dunkirk - [ ] Full Metal Jacket - [ ] Come and See - [ ] Shoah - [ ] Paths of Glory - [ ] Apocalypse Now - [ ] Downfall - [ ] Platoon - [ ] The Deer Hunter - [ ] The Empire of the Sun
/Users/amit/Obsidian/Google Keep/War_Holocaust films.md
0
----
‌RULE 7 Pursue what is meaningful (not what is expedient).md
of the populace was wrong, even if many barbaric practices still existed. It objected to infanticide, to prostitution, and to the principle that might means right. It insisted that women were as valuable as men (even though we are still working out how to manifest that insistence pol